# Fine-Tune Evidence-Aware RAG Synthesizer

This notebook uses QLoRA (4-bit quantization) to fine-tune `Qwen1.5-0.5B-Chat` to act as an advanced, hallucination-free RAG Synthesizer. It is designed specifically to fit on a 4GB VRAM GPU (like a GTX 1650 Ti).

In [1]:
# Install required packages into this specific notebook kernel
%pip install datasets trl peft bitsandbytes

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

os.environ['CUDA_MODULE_LOADING'] = 'EAGER'
if torch.cuda.is_available():
    print(f'CUDA: {torch.cuda.get_device_name(0)} | Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')


CUDA: NVIDIA GeForce GTX 1650 Ti | Memory: 4.29 GB


### Load Dataset

In [3]:
# Load our custom generated dataset
dataset = load_dataset('json', data_files={'train': 'D:/RES_cove/adaptive-evidence-rag/data/finetune/train.jsonl', 'val': 'D:/RES_cove/adaptive-evidence-rag/data/finetune/val.jsonl'})
print(dataset)

def formatting_prompts_func(example):
    if isinstance(example['instruction'], str):
        # Return a single string for a single un-batched example
        return f"{example['instruction']}\n{example['output']}<|im_end|>"
    # Return a list of strings for a batch
    output_texts = []
    for i in range(len(example['instruction'])):
        text = f"{example['instruction'][i]}\n{example['output'][i]}<|im_end|>"
        output_texts.append(text)
    return output_texts


DatasetDict({
    train: Dataset({
        features: ['instruction', 'output'],
        num_rows: 1800
    })
    val: Dataset({
        features: ['instruction', 'output'],
        num_rows: 200
    })
})


### Load Model & Quantization

In [4]:
model_name = "Qwen/Qwen1.5-0.5B-Chat"
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)
print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
print("Loading 4-bit Model...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
print(f"Model dtype: {model.dtype}")
# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)



Loading Tokenizer...


Loading 4-bit Model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model dtype: torch.float16


### LoRA Configuration

In [5]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


trainable params: 3,784,704 || all params: 467,772,416 || trainable%: 0.8091


### Custom Training Loop (GradScaler-Free)

Using a manual loop to avoid BFloat16 GradScaler crash on GTX 1650 Ti.


In [7]:
import os
# Formatting function already defined in dataset cell above
def tokenize_function(examples):
    texts = [f"{inst}{out}" for inst, out in zip(examples['instruction'], examples['output'])]
    return tokenizer(texts, truncation=True, max_length=1024, padding='max_length')
print("Tokenizing datasets...")
tokenized_train = dataset['train'].map(tokenize_function, batched=True, remove_columns=dataset['train'].column_names)
tokenized_val = dataset['val'].map(tokenize_function, batched=True, remove_columns=dataset['val'].column_names)
tokenized_train.set_format(type='torch', columns=['input_ids', 'attention_mask'])
tokenized_val.set_format(type='torch', columns=['input_ids', 'attention_mask'])
# DataLoaders
train_loader = DataLoader(tokenized_train, batch_size=1, shuffle=True)
val_loader = DataLoader(tokenized_val, batch_size=1, shuffle=False)
# Optimizer and scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
num_training_steps = 500
warmup_steps = 15
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=num_training_steps)
# Training loop
print("Starting training...")
model.train()
step = 0
epoch = 0
accumulation_steps = 4
while step < num_training_steps:
    epoch += 1
    epoch_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}")
    
    for batch in pbar:
        if step >= num_training_steps:
            break
        
        input_ids = batch['input_ids'].to(model.device)
        attention_mask = batch['attention_mask'].to(model.device)
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100  # ignore padding in loss
        
        # Forward pass with fp16 autocast (NO GradScaler!)
        with torch.amp.autocast('cuda', dtype=torch.float16):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss / accumulation_steps
        
        # Backward pass
        loss.backward()
        
        epoch_loss += loss.item() * accumulation_steps
        
        if (step + 1) % accumulation_steps == 0:
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.3)
            # Optimizer step
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        
        step += 1
        pbar.set_postfix({'loss': f'{loss.item() * accumulation_steps:.4f}', 'step': f'{step}/{num_training_steps}'})
        
        # Logging
        if step % 10 == 0:
            print(f"Step {step}/{num_training_steps} | Loss: {loss.item() * accumulation_steps:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")
        
        # Save checkpoint
        if step % 100 == 0:
            save_path = f"../models/qwen-evidence-rag-lora/checkpoint-{step}"
            os.makedirs(save_path, exist_ok=True)
            model.save_pretrained(save_path)
            print(f"Checkpoint saved to {save_path}")
    
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch} complete. Average loss: {avg_loss:.4f}")
print(f"Training complete after {step} steps!")



Tokenizing datasets...


Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Starting training...


Epoch 1:   0%|          | 0/1800 [00:00<?, ?it/s][transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
c:\Users\dynos\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\_dynamo\eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
Epoch 1:   1%|          | 10/1800 [01:21<3:58:33,  8.00s/it, loss=4.0237, step=10/500]

...

Step 490/500 | Loss: 3.2116 | LR: 1.56e-04


Epoch 1:  28%|██▊       | 499/1800 [1:18:00<3:34:21,  9.89s/it, loss=2.6181, step=500/500]

Step 500/500 | Loss: 2.6181 | LR: 1.55e-04


Epoch 1:  28%|██▊       | 500/1800 [1:18:01<3:22:52,  9.36s/it, loss=2.6181, step=500/500]

Checkpoint saved to ../models/qwen-evidence-rag-lora/checkpoint-500
Epoch 1 complete. Average loss: 0.8354
Training complete after 500 steps!


In [8]:
# Save the final LoRA adapters
output_dir = "../models/qwen-evidence-rag-lora-final"
os.makedirs(output_dir, exist_ok=True)
model.save_pretrained(output_dir)
print(f"Model saved to {output_dir}")



Model saved to ../models/qwen-evidence-rag-lora-final
